# Peer REIT Financial Data Cleaning

## BSMM 8730 - Group 2 Assignment

### Companies
- RioCan REIT
- Choice Properties REIT

### Objective
Extract, clean, standardize, and validate financial
and operational metrics for peer comparison with
SmartCentres REIT.

### Reporting Period
- Annual: 2023–2025
- Quarterly: Q1–Q3 2026, where available

### Methodology
- PDF text extraction
- Financial metric identification
- Semi-automated data extraction
- Unit and metric standardization
- Data quality validation
- CSV export

In [2]:
# Step 1 - Import required libraries

import pandas as pd
import numpy as np
import pdfplumber
import re

from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.float_format", "{:,.3f}".format)

print("Libraries imported successfully")

Libraries imported successfully


In [10]:
# Step 2 - Locate peer financial reports

base_dir = Path.home() / "Desktop" / "8730 _ Group Assignment"

riocan_dir = base_dir / "RIO Report"

choice_dir = base_dir / "Choice Properties Annual report"

riocan_files = sorted(riocan_dir.rglob("*.pdf"))

choice_files = sorted(choice_dir.rglob("*.pdf"))

peer_files = riocan_files + choice_files

print("RioCan PDF Files:", len(riocan_files))
print("Choice Properties PDF Files:", len(choice_files))
print("Total Peer PDF Files:", len(peer_files))

for file in peer_files:
    print(file.relative_to(base_dir))

RioCan PDF Files: 6
Choice Properties PDF Files: 7
Total Peer PDF Files: 13
RIO Report/REI-Q4-2024-Report-to-Unitholders.pdf
RIO Report/REI-Q4-2025-Report-to-Unitholders.pdf
RIO Report/REI_Q4-2023-Annual-Report.pdf
RIO Report/未命名文件夹/REI-Q1-2026-Report-to-Unitholders.pdf
RIO Report/未命名文件夹/REI-Q2-2026-Report-to-Unitholders.pdf
RIO Report/未命名文件夹/REI-Q3-2026-Report-to-Unitholders.pdf
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q1 2026 Report.pdf
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q2 2026 Report.pdf
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q3 2026 Report.pdf
Choice Properties Annual report/Choice-Properties-2025-Annual-Report-English-AODA.pdf
Choice Properties Annual report/Choice-REIT-Annual-Report-2023-Final-Release-AODA.pdf
Choice Properties Annual report/Choice_Properties_2023_Financial_Extract.pdf
Choice Properties Annual report/Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf


In [12]:
# Step 2.1 - Check all available PDF files

print("Total PDF Files:", len(pdf_files))

for file in pdf_files:
    print(file.relative_to(base_dir))

Total PDF Files: 19
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q1 2026 Report.pdf
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q2 2026 Report.pdf
Choice Properties Annual report/2026 Q1-Q3/Choice Properties Q3 2026 Report.pdf
Choice Properties Annual report/Choice-Properties-2025-Annual-Report-English-AODA.pdf
Choice Properties Annual report/Choice-REIT-Annual-Report-2023-Final-Release-AODA.pdf
Choice Properties Annual report/Choice_Properties_2023_Financial_Extract.pdf
Choice Properties Annual report/Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf
RIO Report/REI-Q4-2024-Report-to-Unitholders.pdf
RIO Report/REI-Q4-2025-Report-to-Unitholders.pdf
RIO Report/REI_Q4-2023-Annual-Report.pdf
RIO Report/未命名文件夹/REI-Q1-2026-Report-to-Unitholders.pdf
RIO Report/未命名文件夹/REI-Q2-2026-Report-to-Unitholders.pdf
RIO Report/未命名文件夹/REI-Q3-2026-Report-to-Unitholders.pdf
SmartCentres REIT/2023-SmartCentres-REIT-Annual-Report.pdf
SmartCentres REIT/2024-SmartCentres-REIT-Annual-Re

In [8]:
# Step 2.2 - Check project folders

for folder in sorted(base_dir.iterdir()):
    if folder.is_dir():
        print(folder.name)

Choice Properties Annual report
RIO Report
SmartCentres REIT
macroeconomic
market_data
processed
validation


In [14]:
# Step 3 - Inspect peer financial reports

report_inventory = []

for company, files in [
    ("RioCan", riocan_files),
    ("Choice Properties", choice_files)
]:

    for file in files:

        try:

            with pdfplumber.open(file) as pdf:

                total_pages = len(pdf.pages)

                sample_pages = pdf.pages[:10]

                readable_pages = sum(
                    bool((page.extract_text() or "").strip())
                    for page in sample_pages
                )

                report_inventory.append({
                    "company": company,
                    "file_name": file.name,
                    "total_pages": total_pages,
                    "sample_readable_pages": readable_pages,
                    "text_extractable": readable_pages > 0
                })

        except Exception as e:

            print("Error:", file.name, str(e))

inventory_df = pd.DataFrame(report_inventory)

display(inventory_df)

,company,file_name,total_pages,sample_readable_pages,text_extractable
0,RioCan,REI-Q4-2024-Report-to-Unitholders.pdf,165,10,True
1,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,170,10,True
2,RioCan,REI_Q4-2023-Annual-Report.pdf,162,9,True
3,RioCan,REI-Q1-2026-Report-to-Unitholders.pdf,108,10,True
4,RioCan,REI-Q2-2026-Report-to-Unitholders.pdf,117,10,True
5,RioCan,REI-Q3-2026-Report-to-Unitholders.pdf,117,10,True
6,Choice Properties,Choice Properties Q1 2026 Report.pdf,91,10,True
7,Choice Properties,Choice Properties Q2 2026 Report.pdf,98,10,True
8,Choice Properties,Choice Properties Q3 2026 Report.pdf,99,10,True
9,Choice Properties,Choice-Properties-2025-Annual-Report-English-AODA.pdf,156,10,True


In [16]:
# Step 4 - Extract text from all peer financial reports

peer_page_texts = {}
extraction_summary = []

for company, files in [
    ("RioCan", riocan_files),
    ("Choice Properties", choice_files)
]:

    for file in files:

        # Exclude supplementary duplicate report
        if "financial_extract" in file.name.lower():
            continue

        pages = {}

        with pdfplumber.open(file) as pdf:

            for page_number, page in enumerate(
                pdf.pages, start=1
            ):

                text = page.extract_text(
                    layout=True
                ) or ""

                pages[page_number] = text

        peer_page_texts[(company, file.name)] = pages

        readable = sum(
            bool(text.strip())
            for text in pages.values()
        )

        extraction_summary.append({
            "company": company,
            "file_name": file.name,
            "total_pages": len(pages),
            "readable_pages": readable,
            "unreadable_pages": len(pages) - readable,
            "readable_pct": round(
                readable / len(pages) * 100, 2
            )
        })

        print("Completed:", file.name)

extraction_summary_df = pd.DataFrame(
    extraction_summary
)

display(extraction_summary_df)

print("Reports Processed:", len(peer_page_texts))

Completed: REI-Q4-2024-Report-to-Unitholders.pdf
Completed: REI-Q4-2025-Report-to-Unitholders.pdf
Completed: REI_Q4-2023-Annual-Report.pdf
Completed: REI-Q1-2026-Report-to-Unitholders.pdf
Completed: REI-Q2-2026-Report-to-Unitholders.pdf
Completed: REI-Q3-2026-Report-to-Unitholders.pdf
Completed: Choice Properties Q1 2026 Report.pdf
Completed: Choice Properties Q2 2026 Report.pdf
Completed: Choice Properties Q3 2026 Report.pdf
Completed: Choice-Properties-2025-Annual-Report-English-AODA.pdf
Completed: Choice-REIT-Annual-Report-2023-Final-Release-AODA.pdf
Completed: Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf


,company,file_name,total_pages,readable_pages,unreadable_pages,readable_pct
0,RioCan,REI-Q4-2024-Report-to-Unitholders.pdf,165,165,0,100.000
1,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,170,170,0,100.000
2,RioCan,REI_Q4-2023-Annual-Report.pdf,162,161,1,99.380
3,RioCan,REI-Q1-2026-Report-to-Unitholders.pdf,108,107,1,99.070
4,RioCan,REI-Q2-2026-Report-to-Unitholders.pdf,117,116,1,99.150
5,RioCan,REI-Q3-2026-Report-to-Unitholders.pdf,117,116,1,99.150
6,Choice Properties,Choice Properties Q1 2026 Report.pdf,91,91,0,100.000
7,Choice Properties,Choice Properties Q2 2026 Report.pdf,98,98,0,100.000
8,Choice Properties,Choice Properties Q3 2026 Report.pdf,99,99,0,100.000
9,Choice Properties,Choice-Properties-2025-Annual-Report-English-AODA.pdf,156,156,0,100.000


Reports Processed: 12


In [18]:
# Step 5 - Locate financial statements and REIT metrics

financial_keywords = [
    "consolidated balance sheets",
    "consolidated statements of financial position",
    "consolidated statements of income",
    "consolidated statements of earnings",
    "consolidated statements of comprehensive income",
    "consolidated statements of cash flows",
    "total assets",
    "total liabilities",
    "financial highlights",
    "net operating income",
    "funds from operations",
    "adjusted funds from operations",
    "same property",
    "same-property",
    "same asset",
    "occupancy rate",
    "net asset value",
    "interest coverage",
    "debt to assets"
]

keyword_results = []

for (company, file_name), pages in peer_page_texts.items():

    normalized_pages = {
        page: " ".join(text.lower().split())
        for page, text in pages.items()
    }

    for keyword in financial_keywords:

        matched_pages = [
            page
            for page, text in normalized_pages.items()
            if keyword.lower() in text
        ]

        if matched_pages:

            keyword_results.append({
                "company": company,
                "file_name": file_name,
                "keyword": keyword,
                "matched_pages": matched_pages,
                "number_of_pages": len(matched_pages)
            })

peer_keyword_report = pd.DataFrame(keyword_results)

print("Total Keyword Matches:", len(peer_keyword_report))

# Preview 2025 annual reports only

display(
    peer_keyword_report[
        peer_keyword_report["file_name"].str.contains("2025")
    ].reset_index(drop=True)
)

Total Keyword Matches: 144


,company,file_name,keyword,matched_pages,number_of_pages
0,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,consolidated balance sheets,"[68, 80, 81, 113, 115, 118, 123, 124, 125, 126, 128, 130, 131, 147, 159, 160, 163, 164, 168]",19
1,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,consolidated statements of income,"[66, 68, 82, 113, 115, 119, 125, 126, 130, 132, 163]",11
2,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,consolidated statements of comprehensive income,"[113, 115, 120, 126, 132]",5
3,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,consolidated statements of cash flows,"[113, 115, 122]",3
4,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,total assets,"[31, 43, 65, 76, 80, 81, 116, 118, 138, 159]",10
5,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,total liabilities,"[80, 81, 118, 159]",4
6,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,financial highlights,[4],1
7,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,net operating income,"[4, 16, 17, 28, 31, 33, 37, 38, 67, 73, 116, 125, 138, 139]",14
8,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,funds from operations,"[16, 37, 38, 155]",4
9,RioCan,REI-Q4-2025-Report-to-Unitholders.pdf,adjusted funds from operations,"[16, 38]",2


In [20]:
# Step 6 - Check unreadable pages

for (company, file_name), pages in peer_page_texts.items():

    unreadable_pages = [
        page_number
        for page_number, text in pages.items()
        if not text.strip()
    ]

    if unreadable_pages:

        print(company)
        print(file_name)
        print("Unreadable Pages:", unreadable_pages)
        print("-" * 60)

RioCan
REI_Q4-2023-Annual-Report.pdf
Unreadable Pages: [1]
------------------------------------------------------------
RioCan
REI-Q1-2026-Report-to-Unitholders.pdf
Unreadable Pages: [108]
------------------------------------------------------------
RioCan
REI-Q2-2026-Report-to-Unitholders.pdf
Unreadable Pages: [117]
------------------------------------------------------------
RioCan
REI-Q3-2026-Report-to-Unitholders.pdf
Unreadable Pages: [117]
------------------------------------------------------------


In [22]:
# Step 7 - Inspect 2025 consolidated financial statements

annual_2025_pages = {
    "RioCan": [118, 119, 122],
    "Choice Properties": [105, 106, 108]
}

for (company, file_name), pages in peer_page_texts.items():

    if "2025" not in file_name:
        continue

    print("\n" + "=" * 80)
    print("Company:", company)
    print("File:", file_name)

    for page_number in annual_2025_pages[company]:

        print("\n" + "=" * 80)
        print("PDF PAGE:", page_number)
        print("=" * 80)

        print(pages.get(page_number, "Page not found"))


Company: RioCan
File: REI-Q4-2025-Report-to-Unitholders.pdf

PDF PAGE: 118
                                                                                    
                                                                                    
       RIOCAN REAL ESTATE INVESTMENT TRUST                                          
       CONSOLIDATED BALANCE SHEETS                                                  
       (In thousands of Canadian dollars)                                           
                                                                                    
        As at                                  Note December 31, 2025 December 31, 2024
        Assets                                                                      
        Investment properties                   3 $     13,628,959 $  13,839,154    
        Equity-accounted investments            4         159,596      408,588      
        Residential inventory                   5         236,745      

In [24]:
# Step 8 - Universal financial extraction function

import re
import pandas as pd
import numpy as np

def parse_financial_number(value):

    value = str(value).strip()

    if value in ["—", "–", "-"]:
        return 0.0

    value = value.replace("$", "").replace(",", "").strip()

    if value.startswith("(") and value.endswith(")"):
        return -float(value[1:-1])

    return float(value)


def extract_peer_values(text, label, years):

    matches = []

    number_pattern = (
        r"\(?-?\d[\d,]*(?:\.\d+)?\)?|[—–]"
    )

    for line in text.splitlines():

        clean_line = " ".join(line.split())

        if not clean_line.lower().startswith(label.lower()):
            continue

        remaining = clean_line[len(label):].strip()

        # Exclude other metrics with similar names
        if remaining and remaining[0].isalpha():
            continue

        numbers = re.findall(
            number_pattern,
            remaining
        )

        if len(numbers) >= len(years):

            values = [
                parse_financial_number(n)
                for n in numbers[-len(years):]
            ]

            matches.append({
                "values": dict(zip(years, values)),
                "original_line": clean_line
            })

    # Remove identical repeated rows
    unique_matches = {}

    for match in matches:

        key = tuple(match["values"].values())
        unique_matches[key] = match

    matches = list(unique_matches.values())

    if len(matches) != 1:

        raise ValueError(
            f"{label}: Expected one row, "
            f"found {len(matches)}"
        )

    return matches[0]


print("Universal Extraction Function: Ready")

Universal Extraction Function: Ready


In [26]:
# Step 9 - Test universal extraction function

annual_2025_reports = {
    company: {
        "file_name": file_name,
        "pages": pages
    }
    for (company, file_name), pages
    in peer_page_texts.items()
    if "2025" in file_name
}

rio_pages = annual_2025_reports["RioCan"]["pages"]

choice_pages = annual_2025_reports[
    "Choice Properties"
]["pages"]

rio_test = extract_peer_values(
    rio_pages[118],
    "Total assets",
    [2025, 2024]
)

choice_test = extract_peer_values(
    choice_pages[105],
    "Total Assets",
    [2025, 2024]
)

print("RioCan:")
print(rio_test)

print("\nChoice Properties:")
print(choice_test)

RioCan:
{'values': {2025: 14894392.0, 2024: 15472044.0}, 'original_line': 'Total assets $ 14,894,392 $ 15,472,044'}

Choice Properties:
{'values': {2025: 17913146.0, 2024: 17557532.0}, 'original_line': 'Total Assets $ 17,913,146 $ 17,557,532'}


In [28]:
# Step 10 - Define peer financial metric mapping

peer_metric_mapping = {

    "RioCan": {

        "total_assets": (118, "Total assets"),
        "total_liabilities": (118, "Total liabilities"),
        "total_equity": (118, "Unitholders' equity"),

        "investment_properties": (
            118, "Investment properties"
        ),

        "cash_and_cash_equivalents": (
            118, "Cash and cash equivalents"
        ),

        "debentures_payable": (
            118, "Debentures payable"
        ),

        "mortgages_payable": (
            118, "Mortgages payable"
        ),

        "mortgages_held_for_sale": (
            118,
            "Mortgages payable associated with assets held for sale"
        ),

        "bank_loans": (
            118, "Lines of credit and other bank loans"
        ),

        "rental_revenue": (119, "Rental revenue"),

        "recoverable_operating_costs": (
            119, "Recoverable under tenant leases"
        ),

        "non_recoverable_operating_costs": (
            119, "Non-recoverable costs"
        ),

        "net_income": (119, "Net income"),

        "interest_costs_net": (
            119, "Interest costs, net"
        ),

        "operating_cash_flow": (
            122, "Cash provided by operating activities"
        ),

        "investing_cash_flow": (
            122, "Cash provided by (used in) investing activities"
        ),

        "financing_cash_flow": (
            122, "Cash provided by (used in) financing activities"
        )
    },

    "Choice Properties": {

        "total_assets": (105, "Total Assets"),
        "total_liabilities": (105, "Total Liabilities"),
        "total_equity": (105, "Total Equity"),

        "investment_properties": (
            105, "Investment properties"
        ),

        "cash_and_cash_equivalents": (
            105, "Cash and cash equivalents"
        ),

        "total_debt": (105, "Long term debt"),

        "exchangeable_units_liability": (
            105, "Exchangeable Units"
        ),

        "rental_revenue": (106, "Rental revenue"),

        "property_operating_costs": (
            106, "Property operating costs"
        ),

        "net_income": (106, "Net (Loss) Income"),

        "net_financing_charges": (
            106,
            "Net interest expense and other financing charges"
        ),

        "operating_cash_flow": (
            108, "Cash Flows from Operating Activities"
        ),

        "investing_cash_flow": (
            108, "Cash Flows used in Investing Activities"
        ),

        "financing_cash_flow": (
            108, "Cash Flows used in Financing Activities"
        )
    }
}

for company, mapping in peer_metric_mapping.items():

    print(company, "Mapped Metrics:", len(mapping))

RioCan Mapped Metrics: 17
Choice Properties Mapped Metrics: 14


In [30]:
# Step 11 - Batch extract annual financial data

peer_tickers = {
    "RioCan": "REI-UN.TO",
    "Choice Properties": "CHP-UN.TO"
}

peer_records = []
extraction_errors = []

positive_expense_metrics = [
    "recoverable_operating_costs",
    "non_recoverable_operating_costs",
    "interest_costs_net"
]

for company, mapping in peer_metric_mapping.items():

    report = annual_2025_reports[company]

    for metric, (page, label) in mapping.items():

        try:

            result = extract_peer_values(
                report["pages"][page],
                label,
                [2025, 2024]
            )

            for year, original_value in result["values"].items():

                value = original_value / 1000

                # Standardize expense signs
                if metric in positive_expense_metrics:
                    value = -abs(value)

                peer_records.append({

                    "company": company,

                    "ticker": peer_tickers[company],

                    "period_end": f"{year}-12-31",

                    "period_type": "FY",

                    "metric": metric,

                    "value": value,

                    "unit": "CAD million",

                    "source_file": report["file_name"],

                    "source_page": page,

                    "source_label": label,

                    "original_value": original_value,

                    "original_unit": "CAD thousand",

                    "extraction_method": "Text-based extraction",

                    "original_line": result["original_line"]

                })

        except Exception as e:

            extraction_errors.append({
                "company": company,
                "metric": metric,
                "source_page": page,
                "error": str(e)
            })

peer_annual_financial = pd.DataFrame(peer_records)

peer_extraction_errors = pd.DataFrame(
    extraction_errors
)

print("Extracted Shape:", peer_annual_financial.shape)

print("Extraction Errors:", len(extraction_errors))

if extraction_errors:
    display(peer_extraction_errors)

with pd.option_context("display.float_format", "{:,.3f}".format):

    display(
        peer_annual_financial.pivot(
            index=["company", "metric"],
            columns="period_end",
            values="value"
        )
    )

Extracted Shape: (62, 14)
Extraction Errors: 0


period_end                                         2024-12-31  2025-12-31
company           metric                                                 
Choice Properties cash_and_cash_equivalents            63.388      71.180
                  exchangeable_units_liability      5,283.750   5,861.598
                  financing_cash_flow                -329.557    -139.454
                  investing_cash_flow                -584.208    -550.460
                  investment_properties            15,331.000  15,758.000
                  net_financing_charges              -586.388    -597.632
                  net_income                          784.437     -61.188
                  operating_cash_flow                 724.729     697.706
                  property_operating_costs           -381.568    -399.144
                  rental_revenue                    1,358.105   1,414.621
                  total_assets                     17,557.532  17,913.146
                  total_debt                        6,684.940   6,804.960
                  total_equity                      4,899.800   4,584.809
                  total_liabilities                12,657.732  13,328.337
RioCan            bank_loans                          383.658     601.194
                  cash_and_cash_equivalents           190.243     145.040
                  debentures_payable                4,088.654   4,338.865
                  financing_cash_flow                  48.495    -520.378
                  interest_costs_net                 -257.544    -277.885
                  investing_cash_flow                -360.766      41.107
                  investment_properties            13,839.154  13,628.959
                  mortgages_held_for_sale               0.000      28.343
                  mortgages_payable                 2,851.602   2,184.306
                  net_income                          473.465      69.295
                  non_recoverable_operating_costs     -37.147     -41.638
                  operating_cash_flow                 378.280     434.068
                  recoverable_operating_costs        -397.042    -414.386
                  rental_revenue                    1,137.127   1,176.428
                  total_assets                     15,472.044  14,894.392
                  total_equity                      7,558.338   7,157.263
                  total_liabilities                 7,913.706   7,737.129

In [32]:
# Step 12 - Calculate derived financial metrics

derived_records = []

for (company, period_end), group in \
        peer_annual_financial.groupby(
            ["company", "period_end"]
        ):

    values = group.set_index("metric")["value"]

    template = group.iloc[0].to_dict()

    derived_metrics = {}

    if company == "RioCan":

        total_debt = (
            values["debentures_payable"]
            + values["mortgages_payable"]
            + values["mortgages_held_for_sale"]
            + values["bank_loans"]
        )

        operating_costs = (
            values["recoverable_operating_costs"]
            + values["non_recoverable_operating_costs"]
        )

        derived_metrics["total_debt"] = (
            total_debt,
            "CAD million",
            "debentures_payable + mortgages_payable + mortgages_held_for_sale + bank_loans"
        )

        derived_metrics["property_operating_costs"] = (
            operating_costs,
            "CAD million",
            "recoverable_operating_costs + non_recoverable_operating_costs"
        )

    else:

        total_debt = values["total_debt"]

        operating_costs = values[
            "property_operating_costs"
        ]

    net_rental_income = (
        values["rental_revenue"]
        + operating_costs
    )

    derived_metrics["net_rental_income"] = (
        net_rental_income,
        "CAD million",
        "rental_revenue + property_operating_costs"
    )

    derived_metrics["book_debt_to_assets_ratio"] = (
        total_debt / values["total_assets"] * 100,
        "Percent",
        "total_debt / total_assets * 100"
    )

    derived_metrics["net_rental_income_margin"] = (
        net_rental_income
        / values["rental_revenue"] * 100,
        "Percent",
        "net_rental_income / rental_revenue * 100"
    )

    for metric, (value, unit, formula) in \
            derived_metrics.items():

        record = template.copy()

        record.update({
            "metric": metric,
            "value": value,
            "unit": unit,
            "source_page": pd.NA,
            "source_label": formula,
            "original_value": np.nan,
            "original_unit": pd.NA,
            "extraction_method": "Calculated",
            "original_line": formula
        })

        derived_records.append(record)

peer_derived = pd.DataFrame(derived_records)

print("Derived Shape:", peer_derived.shape)

display(
    peer_derived.pivot(
        index=["company", "metric"],
        columns="period_end",
        values="value"
    ).round(3)
)

Derived Shape: (16, 14)


period_end                                   2024-12-31  2025-12-31
company           metric                                           
Choice Properties book_debt_to_assets_ratio      38.074      37.989
                  net_rental_income             976.537   1,015.477
                  net_rental_income_margin       71.904      71.784
RioCan            book_debt_to_assets_ratio      47.336      48.023
                  net_rental_income             702.938     720.404
                  net_rental_income_margin       61.817      61.237
                  property_operating_costs     -434.189    -456.024
                  total_debt                  7,323.914   7,152.708

In [34]:
# Step 13 - Combine and validate financial data

peer_annual_combined = pd.concat(
    [
        peer_annual_financial,
        peer_derived
    ],
    ignore_index=True
)

primary_key = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

print("Combined Shape:", peer_annual_combined.shape)

print(
    "Duplicate Primary Keys:",
    peer_annual_combined.duplicated(primary_key).sum()
)

print(
    "Missing Financial Values:",
    peer_annual_combined["value"].isna().sum()
)

for (company, period_end), group in \
        peer_annual_combined.groupby(
            ["company", "period_end"]
        ):

    values = group.set_index("metric")["value"]

    print("=" * 60)
    print("Company:", company)
    print("Period:", period_end)

    print(
        "Balance Sheet Check:",
        np.isclose(
            values["total_assets"],
            values["total_liabilities"]
            + values["total_equity"],
            atol=0.001,
            rtol=0
        )
    )

    print(
        "Net Rental Income Check:",
        np.isclose(
            values["net_rental_income"],
            values["rental_revenue"]
            + values["property_operating_costs"],
            atol=0.001,
            rtol=0
        )
    )

print("\nRecords per Company and Year:")

display(
    peer_annual_combined.groupby(
        ["company", "period_end"]
    ).size()
)

Combined Shape: (78, 14)
Duplicate Primary Keys: 0
Missing Financial Values: 0
Company: Choice Properties
Period: 2024-12-31
Balance Sheet Check: True
Net Rental Income Check: True
Company: Choice Properties
Period: 2025-12-31
Balance Sheet Check: True
Net Rental Income Check: True
Company: RioCan
Period: 2024-12-31
Balance Sheet Check: True
Net Rental Income Check: True
Company: RioCan
Period: 2025-12-31
Balance Sheet Check: True
Net Rental Income Check: True

Records per Company and Year:


company            period_end
Choice Properties  2024-12-31    17
                   2025-12-31    17
RioCan             2024-12-31    22
                   2025-12-31    22
dtype: int64

In [36]:
# Step 14 - Locate annual financial and REIT metrics

search_keywords = [
    "consolidated balance sheets",
    "consolidated statements of income",
    "consolidated statements of cash flows",
    "financial highlights",
    "total assets",
    "total liabilities",
    "net operating income",
    "same property",
    "same-property",
    "funds from operations",
    "adjusted funds from operations",
    "occupancy rate",
    "committed occupancy",
    "net asset value",
    "distributions per unit",
    "payout ratio",
    "interest coverage",
    "debt to adjusted ebitda",
    "leasing spread",
    "renewal spread"
]

annual_keyword_results = []

for (company, file_name), pages in peer_page_texts.items():

    if "2024" not in file_name and "2025" not in file_name:
        continue

    print("=" * 70)
    print(company)
    print(file_name)

    for keyword in search_keywords:

        matched_pages = [
            page_number
            for page_number, text in pages.items()
            if keyword.lower() in
            " ".join(text.lower().split())
        ]

        annual_keyword_results.append({
            "company": company,
            "file_name": file_name,
            "keyword": keyword,
            "matched_pages": matched_pages
        })

        print(keyword, ":", matched_pages)

annual_keyword_report = pd.DataFrame(
    annual_keyword_results
)

RioCan
REI-Q4-2024-Report-to-Unitholders.pdf
consolidated balance sheets : [70, 81, 82, 109, 111, 114, 120, 121, 122, 124, 126, 127, 142, 153, 154, 157, 159]
consolidated statements of income : [68, 70, 83, 109, 111, 115, 121, 122, 125, 128, 157]
consolidated statements of cash flows : [109, 111, 118]
financial highlights : []
total assets : [34, 47, 67, 78, 81, 82, 112, 114, 133, 153]
total liabilities : [81, 82, 114, 153]
net operating income : [12, 17, 18, 34, 36, 40, 70, 76, 112, 121, 134]
same property : [18, 25, 32, 36, 37, 41, 76, 87, 88]
same-property : []
funds from operations : [4, 17, 40, 41, 42, 149]
adjusted funds from operations : [17, 41]
occupancy rate : [30, 105]
committed occupancy : [7, 19, 25, 30]
net asset value : [8, 25, 26, 31, 50, 149]
distributions per unit : [151]
payout ratio : [4, 12, 18, 21, 22, 25, 34, 40, 41, 42, 66, 67, 77, 78, 90, 91, 92, 93, 94]
interest coverage : [56, 144, 158]
debt to adjusted ebitda : [12, 20, 34, 56, 67, 79, 100, 102]
leasing spre

In [38]:
# Step 15 - Extract 2023 annual financial data

annual_2024_reports = {
    company: {
        "file_name": file_name,
        "pages": pages
    }
    for (company, file_name), pages
    in peer_page_texts.items()
    if "2024" in file_name
}

page_mapping_2024 = {
    "RioCan": {
        118: 114,
        119: 115,
        122: 118
    },
    "Choice Properties": {
        105: 109,
        106: 110,
        108: 112
    }
}

records_2023_2024 = []
errors_2023_2024 = []

for company, mapping in peer_metric_mapping.items():

    report = annual_2024_reports[company]

    for metric, (old_page, label) in mapping.items():

        page = page_mapping_2024[company][old_page]

        try:

            result = extract_peer_values(
                report["pages"][page],
                label,
                [2024, 2023]
            )

            for year, original_value in result["values"].items():

                value = original_value / 1000

                if metric in positive_expense_metrics:
                    value = -abs(value)

                records_2023_2024.append({
                    "company": company,
                    "ticker": peer_tickers[company],
                    "period_end": f"{year}-12-31",
                    "period_type": "FY",
                    "metric": metric,
                    "value": value,
                    "unit": "CAD million",
                    "source_file": report["file_name"],
                    "source_page": page,
                    "source_label": label,
                    "original_value": original_value,
                    "original_unit": "CAD thousand",
                    "extraction_method": "Text-based extraction",
                    "original_line": result["original_line"]
                })

        except Exception as e:

            errors_2023_2024.append({
                "company": company,
                "metric": metric,
                "source_page": page,
                "error": str(e)
            })

peer_financial_2023_2024 = pd.DataFrame(
    records_2023_2024
)

print("Extracted Shape:", peer_financial_2023_2024.shape)

print("Extraction Errors:", len(errors_2023_2024))

if errors_2023_2024:
    display(pd.DataFrame(errors_2023_2024))

if not peer_financial_2023_2024.empty:
    display(
        peer_financial_2023_2024.pivot(
            index=["company", "metric"],
            columns="period_end",
            values="value"
        ).round(3)
    )

Extracted Shape: (56, 14)
Extraction Errors: 3


,company,metric,source_page,error
0,RioCan,mortgages_held_for_sale,114,"Mortgages payable associated with assets held for sale: Expected one row, found 0"
1,RioCan,investing_cash_flow,118,"Cash provided by (used in) investing activities: Expected one row, found 0"
2,Choice Properties,net_income,110,"Net (Loss) Income: Expected one row, found 0"


period_end                                         2023-12-31  2024-12-31
company           metric                                                 
Choice Properties cash_and_cash_equivalents           252.424      63.388
                  exchangeable_units_liability      5,521.222   5,283.750
                  financing_cash_flow                 -92.939    -329.557
                  investing_cash_flow                -361.345    -584.208
                  investment_properties            14,923.000  15,331.000
                  net_financing_charges              -566.147    -586.388
                  operating_cash_flow                 641.972     724.729
                  property_operating_costs           -369.060    -381.568
                  rental_revenue                    1,309.170   1,358.105
                  total_assets                     17,308.727  17,557.532
                  total_debt                        6,695.923   6,684.940
                  total_equity                      4,368.502   4,899.800
                  total_liabilities                12,940.225  12,657.732
RioCan            bank_loans                          879.246     383.658
                  cash_and_cash_equivalents           124.234     190.243
                  debentures_payable                3,240.943   4,088.654
                  financing_cash_flow                -147.366      48.495
                  interest_costs_net                 -208.948    -257.544
                  investment_properties            13,561.718  13,839.154
                  mortgages_payable                 2,740.924   2,851.602
                  net_income                           38.802     473.465
                  non_recoverable_operating_costs     -26.320     -37.147
                  operating_cash_flow                 385.516     378.280
                  recoverable_operating_costs        -374.149    -397.042
                  rental_revenue                    1,091.105   1,137.127
                  total_assets                     14,842.281  15,472.044
                  total_equity                      7,437.770   7,558.338
                  total_liabilities                 7,404.511   7,913.706

In [40]:
# Step 16 - Cross-report validation

keys = ["company", "metric"]

old_2024 = peer_financial_2023_2024.loc[
    peer_financial_2023_2024["period_end"] == "2024-12-31",
    keys + ["value", "unit"]
]

new_2024 = peer_annual_financial.loc[
    peer_annual_financial["period_end"] == "2024-12-31",
    keys + ["value", "unit"]
]

peer_comparison = new_2024.merge(
    old_2024,
    on=keys,
    how="outer",
    suffixes=("_2025_report", "_2024_report"),
    indicator=True,
    validate="one_to_one"
)

peer_comparison["difference"] = (
    peer_comparison["value_2025_report"]
    - peer_comparison["value_2024_report"]
)

peer_comparison["match"] = (
    peer_comparison["_merge"].eq("both")
    & peer_comparison["unit_2025_report"].eq(
        peer_comparison["unit_2024_report"]
    )
    & np.isclose(
        peer_comparison["value_2025_report"],
        peer_comparison["value_2024_report"],
        atol=0.001,
        rtol=0
    )
)

print("Expected Metrics:", 31)

print(
    "Compared Metrics:",
    peer_comparison["_merge"].eq("both").sum()
)

print("Matched Metrics:", peer_comparison["match"].sum())

print(
    "Unmatched Metrics:",
    (~peer_comparison["match"]).sum()
)

print(
    "All Matched:",
    len(peer_comparison) == 31
    and peer_comparison["match"].all()
)

display(peer_comparison)

print("\nRecords Requiring Review:")

display(
    peer_comparison[~peer_comparison["match"]]
)

Expected Metrics: 31
Compared Metrics: 28
Matched Metrics: 28
Unmatched Metrics: 3
All Matched: False


,company,metric,value_2025_report,unit_2025_report,value_2024_report,unit_2024_report,_merge,difference,match
0,Choice Properties,cash_and_cash_equivalents,63.388,CAD million,63.388,CAD million,both,0.000,True
1,Choice Properties,exchangeable_units_liability,"5,283.750",CAD million,"5,283.750",CAD million,both,0.000,True
2,Choice Properties,financing_cash_flow,-329.557,CAD million,-329.557,CAD million,both,0.000,True
3,Choice Properties,investing_cash_flow,-584.208,CAD million,-584.208,CAD million,both,0.000,True
4,Choice Properties,investment_properties,"15,331.000",CAD million,"15,331.000",CAD million,both,0.000,True
5,Choice Properties,net_financing_charges,-586.388,CAD million,-586.388,CAD million,both,0.000,True
6,Choice Properties,net_income,784.437,CAD million,NaN,NaN,left_only,NaN,False
7,Choice Properties,operating_cash_flow,724.729,CAD million,724.729,CAD million,both,0.000,True
8,Choice Properties,property_operating_costs,-381.568,CAD million,-381.568,CAD million,both,0.000,True
9,Choice Properties,rental_revenue,"1,358.105",CAD million,"1,358.105",CAD million,both,0.000,True



Records Requiring Review:


,company,metric,value_2025_report,unit_2025_report,value_2024_report,unit_2024_report,_merge,difference,match
6,Choice Properties,net_income,784.437,CAD million,NaN,NaN,left_only,NaN,False
19,RioCan,investing_cash_flow,-360.766,CAD million,NaN,NaN,left_only,NaN,False
21,RioCan,mortgages_held_for_sale,0.000,CAD million,NaN,NaN,left_only,NaN,False


In [42]:
# Step 17 - Investigate missing financial metrics

review_pages = [
    ("RioCan", 114),
    ("RioCan", 118),
    ("Choice Properties", 110)
]

for company, page in review_pages:

    report = annual_2024_reports[company]

    print("=" * 80)
    print("Company:", company)
    print("File:", report["file_name"])
    print("PDF Page:", page)
    print("=" * 80)

    print(report["pages"][page])
    print("\n")

Company: RioCan
File: REI-Q4-2024-Report-to-Unitholders.pdf
PDF Page: 114
                                                                                    
                                                                                    
       RIOCAN REAL ESTATE INVESTMENT TRUST                                          
       CONSOLIDATED BALANCE SHEETS                                                  
       (In thousands of Canadian dollars)                                           
                                                                                    
        As at                                  Note December 31, 2024 December 31, 2023
        Assets                                                                      
        Investment properties                   3 $     13,839,154 $  13,561,718    
        Equity-accounted investments            4         408,588      383,883      
        Mortgages and loans receivable          6         470,729      28

In [44]:
# Step 18 - Identify alternative financial labels

review_keywords = {
    ("RioCan", 114): [
        "mortgage",
        "held for sale"
    ],
    ("RioCan", 118): [
        "investing"
    ],
    ("Choice Properties", 110): [
        "net income",
        "net loss",
        "net (loss)",
        "net (income)"
    ]
}

for (company, page), keywords in review_keywords.items():

    text = annual_2024_reports[company]["pages"][page]

    print("=" * 70)
    print(company, "| PDF Page:", page)
    print("=" * 70)

    for line in text.splitlines():

        if any(
            keyword.lower() in line.lower()
            for keyword in keywords
        ):
            print(line)

RioCan | PDF Page: 114
        Mortgages and loans receivable          6         470,729      289,533      
        Assets held for sale                    3          16,707       19,075      
        Mortgages payable                      11        2,851,602    2,740,924     
RioCan | PDF Page: 118
        Investing activities                                                        
        Cash used in investing activities                       (360,766) (200,145) 
Choice Properties | PDF Page: 110
       Net Income                              $         784,437 $     796,691      
       Net Income                              $         784,437 $     796,691      


In [46]:
# Step 19 - Recover missing financial records

recovery_mapping = [
    {
        "company": "RioCan",
        "metric": "investing_cash_flow",
        "page": 118,
        "label": "Cash used in investing activities",
        "values": {
            2024: -360766,
            2023: -200145
        }
    },
    {
        "company": "Choice Properties",
        "metric": "net_income",
        "page": 110,
        "label": "Net Income",
        "values": {
            2024: 784437,
            2023: 796691
        }
    }
]

recovered_records = []

for item in recovery_mapping:

    company = item["company"]
    report = annual_2024_reports[company]

    lines = [
        line.strip()
        for line in report["pages"][item["page"]].splitlines()
        if line.strip().lower().startswith(
            item["label"].lower()
        )
    ]

    assert lines, f"Source line missing: {item['metric']}"

    for year, original_value in item["values"].items():

        recovered_records.append({
            "company": company,
            "ticker": peer_tickers[company],
            "period_end": f"{year}-12-31",
            "period_type": "FY",
            "metric": item["metric"],
            "value": original_value / 1000,
            "unit": "CAD million",
            "source_file": report["file_name"],
            "source_page": item["page"],
            "source_label": item["label"],
            "original_value": original_value,
            "original_unit": "CAD thousand",
            "extraction_method":
                "Manual transcription from extracted text",
            "original_line": lines[0]
        })

peer_recovered = pd.DataFrame(recovered_records)

print("Recovered Records:", len(peer_recovered))

display(
    peer_recovered[
        ["company", "period_end", "metric",
         "value", "unit", "source_page"]
    ]
)

Recovered Records: 4


,company,period_end,metric,value,unit,source_page
0,RioCan,2024-12-31,investing_cash_flow,-360.766,CAD million,118
1,RioCan,2023-12-31,investing_cash_flow,-200.145,CAD million,118
2,Choice Properties,2024-12-31,net_income,784.437,CAD million,110
3,Choice Properties,2023-12-31,net_income,796.691,CAD million,110


In [48]:
# Step 20 - Merge recovered financial records

primary_keys = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

peer_financial_2023_2024 = pd.concat(
    [
        peer_financial_2023_2024.loc[
            ~peer_financial_2023_2024.set_index(
                primary_keys
            ).index.isin(
                peer_recovered.set_index(primary_keys).index
            )
        ],
        peer_recovered
    ],
    ignore_index=True
)

print(
    "Updated Shape:",
    peer_financial_2023_2024.shape
)

print(
    "Duplicate Primary Keys:",
    peer_financial_2023_2024.duplicated(
        primary_keys
    ).sum()
)

print(
    "Missing Financial Values:",
    peer_financial_2023_2024["value"].isna().sum()
)

Updated Shape: (60, 14)
Duplicate Primary Keys: 0
Missing Financial Values: 0


In [50]:
# Step 21 - Final cross-report validation

documented_exception = (
    peer_comparison["company"].eq("RioCan")
    & peer_comparison["metric"].eq(
        "mortgages_held_for_sale"
    )
    & peer_comparison["_merge"].eq("left_only")
    & peer_comparison["value_2025_report"].eq(0)
)

matched = peer_comparison["match"]

unexpected = ~(matched | documented_exception)

print("Expected Metrics:", 31)

print("Matched Metrics:", matched.sum())

print(
    "Documented Exceptions:",
    documented_exception.sum()
)

print(
    "Unexpected Mismatches:",
    unexpected.sum()
)

print(
    "Validation Passed:",
    len(peer_comparison) == 31
    and matched.sum() == 30
    and documented_exception.sum() == 1
    and not unexpected.any()
)

display(peer_comparison[~matched])

Expected Metrics: 31
Matched Metrics: 28
Documented Exceptions: 1
Unexpected Mismatches: 2
Validation Passed: False


,company,metric,value_2025_report,unit_2025_report,value_2024_report,unit_2024_report,_merge,difference,match
6,Choice Properties,net_income,784.437,CAD million,NaN,NaN,left_only,NaN,False
19,RioCan,investing_cash_flow,-360.766,CAD million,NaN,NaN,left_only,NaN,False
21,RioCan,mortgages_held_for_sale,0.000,CAD million,NaN,NaN,left_only,NaN,False


In [52]:
# Step 21 Revised - Refresh cross-report validation

keys = ["company", "metric"]

old_2024 = peer_financial_2023_2024.loc[
    peer_financial_2023_2024["period_end"] == "2024-12-31",
    keys + ["value", "unit"]
]

new_2024 = peer_annual_financial.loc[
    peer_annual_financial["period_end"] == "2024-12-31",
    keys + ["value", "unit"]
]

# Rebuild comparison using updated data

peer_comparison = new_2024.merge(
    old_2024,
    on=keys,
    how="outer",
    suffixes=("_2025_report", "_2024_report"),
    indicator=True,
    validate="one_to_one"
)

peer_comparison["difference"] = (
    peer_comparison["value_2025_report"]
    - peer_comparison["value_2024_report"]
)

peer_comparison["match"] = (
    peer_comparison["_merge"].eq("both")
    & peer_comparison["unit_2025_report"].eq(
        peer_comparison["unit_2024_report"]
    )
    & np.isclose(
        peer_comparison["value_2025_report"],
        peer_comparison["value_2024_report"],
        atol=0.001,
        rtol=0
    )
)

# Document the reporting exception

documented_exception = (
    peer_comparison["company"].eq("RioCan")
    & peer_comparison["metric"].eq(
        "mortgages_held_for_sale"
    )
    & peer_comparison["_merge"].eq("left_only")
    & peer_comparison["value_2025_report"].eq(0)
)

matched = peer_comparison["match"]

unexpected = ~(matched | documented_exception)

# Final validation

print("Expected Metrics:", 31)
print("Matched Metrics:", matched.sum())
print("Documented Exceptions:", documented_exception.sum())
print("Unexpected Mismatches:", unexpected.sum())

print(
    "Validation Passed:",
    len(peer_comparison) == 31
    and matched.sum() == 30
    and documented_exception.sum() == 1
    and not unexpected.any()
)

display(peer_comparison[~matched])

Expected Metrics: 31
Matched Metrics: 30
Documented Exceptions: 1
Unexpected Mismatches: 0
Validation Passed: True


,company,metric,value_2025_report,unit_2025_report,value_2024_report,unit_2024_report,_merge,difference,match
21,RioCan,mortgages_held_for_sale,0.000,CAD million,NaN,NaN,left_only,NaN,False


In [54]:
# Step 22 - Batch locate core REIT metrics

core_reit_keywords = [
    "net operating income",
    "same property",
    "same asset",
    "same-property",
    "same-asset",
    "same asset noi",
    "ffo",
    "affo",
    "payout ratio",
    "occupancy",
    "net asset value",
    "nav per unit",
    "distributions",
    "interest coverage",
    "leasing spread",
    "renewal spread"
]

for (company, file_name), pages in peer_page_texts.items():

    if "2024" not in file_name and "2025" not in file_name:
        continue

    print("=" * 80)
    print(company)
    print(file_name)
    print("=" * 80)

    for keyword in core_reit_keywords:

        matched_pages = [
            page
            for page, text in pages.items()
            if keyword.lower() in
            " ".join(text.lower().split())
        ]

        if matched_pages:
            print(f"{keyword}: {matched_pages}")

RioCan
REI-Q4-2024-Report-to-Unitholders.pdf
net operating income: [12, 17, 18, 34, 36, 40, 70, 76, 112, 121, 134]
same property: [18, 25, 32, 36, 37, 41, 76, 87, 88]
ffo: [4, 12, 17, 18, 20, 21, 22, 23, 24, 25, 30, 33, 34, 37, 38, 39, 40, 41, 42, 48, 49, 66, 67, 77, 88, 89, 90, 91, 92, 93, 94, 103, 106, 108]
affo: [17, 18, 24, 30, 34, 41, 42, 49, 67, 77, 93, 94, 103, 106]
payout ratio: [4, 12, 18, 21, 22, 25, 34, 40, 41, 42, 66, 67, 77, 78, 90, 91, 92, 93, 94]
occupancy: [4, 7, 8, 19, 21, 23, 25, 27, 30, 33, 37, 51, 67, 76, 103, 104, 105, 106, 112, 123, 152, 157]
net asset value: [8, 25, 26, 31, 50, 149]
distributions: [12, 17, 18, 21, 25, 34, 40, 41, 42, 61, 62, 65, 66, 67, 77, 78, 80, 90, 91, 93, 94, 118, 119, 128, 137, 143, 146, 147, 148, 149, 150, 151, 158, 160, 163]
interest coverage: [56, 144, 158]
leasing spread: [4, 7, 19, 22, 23, 25, 31, 32, 103]
RioCan
REI-Q4-2025-Report-to-Unitholders.pdf
net operating income: [4, 16, 17, 28, 31, 33, 37, 38, 67, 73, 116, 125, 138, 139]
same

In [56]:
# Step 23 - Extract key financial highlights

highlight_pages = {
    "RioCan": {
        2025: [4, 16, 17, 23, 27, 37, 38, 54],
        2024: [17, 18, 25, 30, 40, 41, 56]
    },
    "Choice Properties": {
        2025: [19, 20, 67, 86, 89, 90, 91],
        2024: [19, 20, 72, 91, 94, 95, 96]
    }
}

for (company, file_name), pages in peer_page_texts.items():

    for year in [2024, 2025]:

        if str(year) not in file_name:
            continue

        print("=" * 80)
        print(company, year)
        print(file_name)

        for page in highlight_pages[company][year]:

            print("\n" + "-" * 70)
            print("PDF PAGE:", page)
            print("-" * 70)

            print(pages.get(page, "Page not found"))

RioCan 2024
REI-Q4-2024-Report-to-Unitholders.pdf

----------------------------------------------------------------------
PDF PAGE: 17
----------------------------------------------------------------------
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                                                                                    
                                                                                    
       TABLE     OF   CONTENT

In [58]:
# Step 24 - Build core peer REIT dataset

peer_reit_records = []

report_files = {
    ("RioCan", 2024):
        "REI-Q4-2024-Report-to-Unitholders.pdf",

    ("RioCan", 2025):
        "REI-Q4-2025-Report-to-Unitholders.pdf",

    ("Choice Properties", 2024):
        "Q30_7336_CP_AR_2024_ENG_March_17A_AODA.pdf",

    ("Choice Properties", 2025):
        "Choice-Properties-2025-Annual-Report-English-AODA.pdf"
}

peer_tickers = {
    "RioCan": "REI-UN.TO",
    "Choice Properties": "CHP-UN.TO"
}


def add_peer_reit(company, report_year, page,
                  metric, label, unit, values):

    for year, original_value in values.items():

        if unit == "CAD million":
            value = original_value / 1000
            original_unit = "CAD thousand"
        else:
            value = original_value
            original_unit = unit

        peer_reit_records.append({
            "company": company,
            "ticker": peer_tickers[company],
            "period_end": f"{year}-12-31",
            "period_type": "FY",
            "metric": metric,
            "value": value,
            "unit": unit,
            "source_file": report_files[
                (company, report_year)
            ],
            "source_page": page,
            "source_label": label,
            "original_value": original_value,
            "original_unit": original_unit,
            "extraction_method":
                "Manual transcription from extracted text",
            "original_line": pd.NA
        })


# RioCan - 2024 Annual Report

rio_2024 = [
    (40, "ffo", "FFO",
     "CAD million", {2023: 531281, 2024: 535971}),

    (40, "ffo_per_unit_diluted",
     "FFO per unit - diluted",
     "CAD/unit", {2023: 1.77, 2024: 1.78}),

    (41, "affo", "AFFO",
     "CAD million", {2023: 459483, 2024: 455995}),

    (41, "affo_per_unit_diluted",
     "AFFO per unit - diluted",
     "CAD/unit", {2023: 1.53, 2024: 1.52}),

    (41, "affo_payout_ratio", "AFFO Payout Ratio",
     "Percent", {2023: 70.0, 2024: 72.8}),

    (30, "occupancy_rate_committed",
     "Total Commercial Committed Occupancy",
     "Percent", {2023: 97.4, 2024: 98.0}),

    (30, "occupancy_rate_in_place",
     "Total Commercial In-Place Occupancy",
     "Percent", {2023: 97.1, 2024: 97.4}),

    (18, "same_property_noi_growth",
     "Commercial SPNOI Growth",
     "Percent", {2024: 1.2})
]


# RioCan - 2025 Annual Report

rio_2025 = [
    (37, "ffo", "FFO",
     "CAD million", {2024: 535971, 2025: 553161}),

    (37, "ffo_per_unit_diluted",
     "FFO per unit - diluted",
     "CAD/unit", {2024: 1.78, 2025: 1.87}),

    (38, "affo", "AFFO",
     "CAD million", {2024: 455995, 2025: 473929}),

    (38, "affo_per_unit_diluted",
     "AFFO per unit - diluted",
     "CAD/unit", {2024: 1.52, 2025: 1.60}),

    (38, "affo_payout_ratio", "AFFO Payout Ratio",
     "Percent", {2024: 72.8, 2025: 71.9}),

    (27, "occupancy_rate_committed",
     "Total Commercial Committed Occupancy",
     "Percent", {2024: 98.0, 2025: 97.8}),

    (27, "occupancy_rate_in_place",
     "Total Commercial In-Place Occupancy",
     "Percent", {2024: 97.4, 2025: 97.0}),

    (17, "same_property_noi_growth",
     "Commercial SPNOI Growth",
     "Percent", {2025: 3.6})
]


# Choice Properties - 2024 Annual Report

choice_2024 = [
    (95, "noi_proportionate",
     "Net Operating Income, Accounting Basis",
     "CAD million", {2023: 995006, 2024: 1036757}),

    (72, "ffo", "Funds from Operations",
     "CAD million", {2023: 726134, 2024: 746770}),

    (72, "ffo_per_unit_diluted",
     "FFO per unit diluted",
     "CAD/unit", {2023: 1.003, 2024: 1.032}),

    (72, "affo", "Adjusted Funds from Operations",
     "CAD million", {2023: 598432, 2024: 624948}),

    (72, "affo_per_unit_diluted",
     "AFFO per unit diluted",
     "CAD/unit", {2023: 0.827, 2024: 0.864}),

    (72, "affo_payout_ratio",
     "AFFO payout ratio - diluted",
     "Percent", {2023: 90.5, 2024: 87.8}),

    (72, "distributions_per_unit",
     "Distribution declared per unit",
     "CAD/unit", {2023: 0.749, 2024: 0.758}),

    (19, "occupancy_rate_reported",
     "Period end occupancy",
     "Percent", {2024: 97.6}),

    (20, "same_property_noi_growth",
     "Same-Asset NOI on a cash basis growth",
     "Percent", {2024: 3.2}),

    (20, "interest_coverage_ratio",
     "Interest Coverage ratio",
     "Times", {2024: 3.3})
]


# Choice Properties - 2025 Annual Report

choice_2025 = [
    (90, "noi_proportionate",
     "Net Operating Income, Accounting Basis",
     "CAD million", {2024: 1036757, 2025: 1087479}),

    (67, "ffo", "Funds from Operations",
     "CAD million", {2024: 746770, 2025: 773844}),

    (67, "ffo_per_unit_diluted",
     "FFO per unit diluted",
     "CAD/unit", {2024: 1.032, 2025: 1.069}),

    (67, "affo", "Adjusted Funds from Operations",
     "CAD million", {2024: 624948, 2025: 631735}),

    (67, "affo_per_unit_diluted",
     "AFFO per unit diluted",
     "CAD/unit", {2024: 0.864, 2025: 0.873}),

    (67, "affo_payout_ratio",
     "AFFO payout ratio - diluted",
     "Percent", {2024: 87.8, 2025: 88.0}),

    (67, "distributions_per_unit",
     "Distribution declared per unit",
     "CAD/unit", {2024: 0.758, 2025: 0.768}),

    (20, "occupancy_rate_reported",
     "Period end occupancy",
     "Percent", {2025: 98.2}),

    (20, "same_property_noi_growth",
     "Same-Asset NOI on a cash basis growth",
     "Percent", {2025: 2.2}),

    (20, "interest_coverage_ratio",
     "Interest Coverage ratio",
     "Times", {2025: 3.2})
]


# Batch processing

for company, report_year, rows in [
    ("RioCan", 2024, rio_2024),
    ("RioCan", 2025, rio_2025),
    ("Choice Properties", 2024, choice_2024),
    ("Choice Properties", 2025, choice_2025)
]:

    for page, metric, label, unit, values in rows:

        add_peer_reit(
            company, report_year, page,
            metric, label, unit, values
        )


peer_reit_raw = pd.DataFrame(peer_reit_records)

print("Raw REIT Shape:", peer_reit_raw.shape)

display(
    peer_reit_raw[
        ["company", "period_end", "metric",
         "value", "unit", "source_page"]
    ]
)

Raw REIT Shape: (64, 14)


,company,period_end,metric,value,unit,source_page
0,RioCan,2023-12-31,ffo,531.281,CAD million,40
1,RioCan,2024-12-31,ffo,535.971,CAD million,40
2,RioCan,2023-12-31,ffo_per_unit_diluted,1.770,CAD/unit,40
3,RioCan,2024-12-31,ffo_per_unit_diluted,1.780,CAD/unit,40
4,RioCan,2023-12-31,affo,459.483,CAD million,41
...,...,...,...,...,...,...
59,Choice Properties,2024-12-31,distributions_per_unit,0.758,CAD/unit,67
60,Choice Properties,2025-12-31,distributions_per_unit,0.768,CAD/unit,67
61,Choice Properties,2025-12-31,occupancy_rate_reported,98.200,Percent,20
62,Choice Properties,2025-12-31,same_property_noi_growth,2.200,Percent,20


In [60]:
# Step 25 - Validate and deduplicate REIT records

import numpy as np

primary_keys = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

reit_cross_validation = (
    peer_reit_raw
    .groupby(primary_keys)
    .agg(
        records=("value", "size"),
        min_value=("value", "min"),
        max_value=("value", "max"),
        unique_units=("unit", "nunique")
    )
    .reset_index()
)

reit_cross_validation["difference"] = (
    reit_cross_validation["max_value"]
    - reit_cross_validation["min_value"]
)

reit_cross_validation["match"] = (
    np.isclose(
        reit_cross_validation["difference"],
        0,
        atol=0.000001,
        rtol=0
    )
    & (reit_cross_validation["unique_units"] == 1)
)

overlapping_metrics = reit_cross_validation[
    reit_cross_validation["records"] > 1
]

conflicts = reit_cross_validation[
    ~reit_cross_validation["match"]
]

print("Compared Metrics:", len(overlapping_metrics))

print(
    "Matched Metrics:",
    overlapping_metrics["match"].sum()
)

print("Conflicts:", len(conflicts))

display(conflicts)

assert len(overlapping_metrics) == 14

assert conflicts.empty, "Resolve conflicts before merging."


# Retain the latest available report

peer_reit_raw["report_year"] = (
    peer_reit_raw["source_file"]
    .str.extract(r"(2024|2025)")[0]
    .astype(int)
)

peer_reit_annual = (
    peer_reit_raw
    .sort_values("report_year")
    .drop_duplicates(
        subset=primary_keys,
        keep="last"
    )
    .drop(columns="report_year")
    .reset_index(drop=True)
)

print("Final REIT Shape:", peer_reit_annual.shape)

print(
    "Duplicate Primary Keys:",
    peer_reit_annual.duplicated(primary_keys).sum()
)

print(
    "Missing Financial Values:",
    peer_reit_annual["value"].isna().sum()
)

display(
    peer_reit_annual.pivot(
        index=["company", "metric"],
        columns="period_end",
        values="value"
    ).round(3)
)

Compared Metrics: 14
Matched Metrics: 14
Conflicts: 0


,ticker,period_end,period_type,metric,records,min_value,max_value,unique_units,difference,match


Final REIT Shape: (50, 14)
Duplicate Primary Keys: 0
Missing Financial Values: 0


period_end                                  2023-12-31  2024-12-31  2025-12-31
company           metric                                                      
Choice Properties affo                         598.432     624.948     631.735
                  affo_payout_ratio             90.500      87.800      88.000
                  affo_per_unit_diluted          0.827       0.864       0.873
                  distributions_per_unit         0.749       0.758       0.768
                  ffo                          726.134     746.770     773.844
                  ffo_per_unit_diluted           1.003       1.032       1.069
                  interest_coverage_ratio          NaN       3.300       3.200
                  noi_proportionate            995.006   1,036.757   1,087.479
                  occupancy_rate_reported          NaN      97.600      98.200
                  same_property_noi_growth         NaN       3.200       2.200
RioCan            affo                         459.483     455.995     473.929
                  affo_payout_ratio             70.000      72.800      71.900
                  affo_per_unit_diluted          1.530       1.520       1.600
                  ffo                          531.281     535.971     553.161
                  ffo_per_unit_diluted           1.770       1.780       1.870
                  occupancy_rate_committed      97.400      98.000      97.800
                  occupancy_rate_in_place       97.100      97.400      97.000
                  same_property_noi_growth         NaN       1.200       3.600

In [62]:
# Step 26 - Supplement key REIT metrics

peer_reit_records = []

supplementary_metrics = [
    (
        "RioCan", 2024, 40,
        "ffo_payout_ratio",
        "FFO Payout Ratio",
        "Percent",
        {2023: 60.5, 2024: 61.9}
    ),
    (
        "RioCan", 2025, 37,
        "ffo_payout_ratio",
        "FFO Payout Ratio",
        "Percent",
        {2025: 61.6}
    ),
    (
        "RioCan", 2024, 56,
        "weighted_average_interest_rate",
        "Weighted average effective interest rate - proportionate share",
        "Percent",
        {2023: 3.87, 2024: 3.99}
    ),
    (
        "RioCan", 2025, 54,
        "weighted_average_interest_rate",
        "Weighted average effective interest rate - proportionate share",
        "Percent",
        {2025: 4.07}
    ),
    (
        "Choice Properties", 2024, 72,
        "ffo_payout_ratio",
        "FFO payout ratio - diluted",
        "Percent",
        {2023: 74.6, 2024: 73.5}
    ),
    (
        "Choice Properties", 2025, 67,
        "ffo_payout_ratio",
        "FFO payout ratio - diluted",
        "Percent",
        {2025: 71.9}
    ),
    (
        "Choice Properties", 2024, 20,
        "adjusted_debt_to_total_assets",
        "Adjusted Debt to Total Assets",
        "Percent",
        {2024: 40.0}
    ),
    (
        "Choice Properties", 2025, 20,
        "adjusted_debt_to_total_assets",
        "Adjusted Debt to Total Assets",
        "Percent",
        {2025: 40.5}
    )
]

for row in supplementary_metrics:
    add_peer_reit(*row)

peer_reit_supplementary = pd.DataFrame(
    peer_reit_records
)

print(
    "Supplementary Shape:",
    peer_reit_supplementary.shape
)

display(
    peer_reit_supplementary[
        ["company", "period_end", "metric",
         "value", "unit", "source_page"]
    ]
)

Supplementary Shape: (11, 14)


,company,period_end,metric,value,unit,source_page
0,RioCan,2023-12-31,ffo_payout_ratio,60.500,Percent,40
1,RioCan,2024-12-31,ffo_payout_ratio,61.900,Percent,40
2,RioCan,2025-12-31,ffo_payout_ratio,61.600,Percent,37
3,RioCan,2023-12-31,weighted_average_interest_rate,3.870,Percent,56
4,RioCan,2024-12-31,weighted_average_interest_rate,3.990,Percent,56
5,RioCan,2025-12-31,weighted_average_interest_rate,4.070,Percent,54
6,Choice Properties,2023-12-31,ffo_payout_ratio,74.600,Percent,72
7,Choice Properties,2024-12-31,ffo_payout_ratio,73.500,Percent,72
8,Choice Properties,2025-12-31,ffo_payout_ratio,71.900,Percent,67
9,Choice Properties,2024-12-31,adjusted_debt_to_total_assets,40.000,Percent,20


In [64]:
# Step 27 - Finalize peer REIT metrics

peer_reit_annual_final = pd.concat(
    [
        peer_reit_annual,
        peer_reit_supplementary
    ],
    ignore_index=True
)

peer_reit_annual_final = (
    peer_reit_annual_final
    .sort_values(
        ["company", "period_end", "metric"]
    )
    .reset_index(drop=True)
)

print("Final Shape:", peer_reit_annual_final.shape)

print(
    "Duplicate Primary Keys:",
    peer_reit_annual_final.duplicated(
        primary_keys
    ).sum()
)

print(
    "Missing Financial Values:",
    peer_reit_annual_final["value"].isna().sum()
)

print(
    "Unit Conflicts:",
    (
        peer_reit_annual_final
        .groupby(["company", "metric"])["unit"]
        .nunique() > 1
    ).sum()
)

display(
    peer_reit_annual_final.pivot(
        index=["company", "metric"],
        columns="period_end",
        values="value"
    ).round(3)
)

Final Shape: (61, 14)
Duplicate Primary Keys: 0
Missing Financial Values: 0
Unit Conflicts: 0


period_end                                        2023-12-31  2024-12-31  \
company           metric                                                   
Choice Properties adjusted_debt_to_total_assets          NaN      40.000   
                  affo                               598.432     624.948   
                  affo_payout_ratio                   90.500      87.800   
                  affo_per_unit_diluted                0.827       0.864   
                  distributions_per_unit               0.749       0.758   
                  ffo                                726.134     746.770   
                  ffo_payout_ratio                    74.600      73.500   
                  ffo_per_unit_diluted                 1.003       1.032   
                  interest_coverage_ratio                NaN       3.300   
                  noi_proportionate                  995.006   1,036.757   
                  occupancy_rate_reported                NaN      97.600   
                  same_property_noi_growth               NaN       3.200   
RioCan            affo                               459.483     455.995   
                  affo_payout_ratio                   70.000      72.800   
                  affo_per_unit_diluted                1.530       1.520   
                  ffo                                531.281     535.971   
                  ffo_payout_ratio                    60.500      61.900   
                  ffo_per_unit_diluted                 1.770       1.780   
                  occupancy_rate_committed            97.400      98.000   
                  occupancy_rate_in_place             97.100      97.400   
                  same_property_noi_growth               NaN       1.200   
                  weighted_average_interest_rate       3.870       3.990   

period_end                                        2025-12-31  
company           metric                                      
Choice Properties adjusted_debt_to_total_assets       40.500  
                  affo                               631.735  
                  affo_payout_ratio                   88.000  
                  affo_per_unit_diluted                0.873  
                  distributions_per_unit               0.768  
                  ffo                                773.844  
                  ffo_payout_ratio                    71.900  
                  ffo_per_unit_diluted                 1.069  
                  interest_coverage_ratio              3.200  
                  noi_proportionate                1,087.479  
                  occupancy_rate_reported             98.200  
                  same_property_noi_growth             2.200  
RioCan            affo                               473.929  
                  affo_payout_ratio                   71.900  
                  affo_per_unit_diluted                1.600  
                  ffo                                553.161  
                  ffo_payout_ratio                    61.600  
                  ffo_per_unit_diluted                 1.870  
                  occupancy_rate_committed            97.800  
                  occupancy_rate_in_place             97.000  
                  same_property_noi_growth             3.600  
                  weighted_average_interest_rate       4.070

In [66]:
# Step 28 - Integrate annual peer financial data

import pandas as pd
import numpy as np

primary_keys = [
    "ticker", "period_end", "period_type", "metric"
]

# Retain 2023 records from the 2024 reports

peer_2023 = peer_financial_2023_2024[
    peer_financial_2023_2024["period_end"]
    .astype(str).eq("2023-12-31")
].copy()

# Combine accounting records

peer_accounting = pd.concat(
    [peer_2023, peer_annual_financial],
    ignore_index=True
)

# Standardize accounting metric names

peer_accounting["metric"] = (
    peer_accounting["metric"].replace({
        "rental_revenue":
            "rental_revenue_and_other",
        "net_rental_income":
            "net_rental_income_and_other"
    })
)

# Recalculate derived metrics consistently

derived_names = [
    "book_debt_to_assets_ratio",
    "net_rental_income_margin",
    "net_rental_income_and_other"
]

peer_accounting = peer_accounting[
    ~peer_accounting["metric"].isin(derived_names)
].copy()

# Recalculate RioCan debt and operating costs

rio_derived_names = [
    "total_debt",
    "property_operating_costs"
]

peer_accounting = peer_accounting[
    ~(
        peer_accounting["company"].eq("RioCan")
        & peer_accounting["metric"].isin(
            rio_derived_names
        )
    )
].copy()

derived_records = []

for (company, period), group in peer_accounting.groupby(
    ["company", "period_end"]
):

    assert not group["metric"].duplicated().any()

    values = group.set_index("metric")["value"].to_dict()
    source_rows = group.set_index("metric")

    def add_derived(metric, value, unit, inputs, formula):

        row = group.iloc[0].to_dict()

        sources = source_rows.loc[inputs]

        row.update({
            "metric": metric,
            "value": value,
            "unit": unit,
            "source_file": sources["source_file"].iloc[0],
            "source_page": (
                sources["source_page"].iloc[0]
                if sources["source_page"].nunique() == 1
                else pd.NA
            ),
            "source_label": formula,
            "original_value": pd.NA,
            "original_unit": pd.NA,
            "original_line": formula,
            "extraction_method": "Calculated"
        })

        derived_records.append(row)
        values[metric] = value

    if company == "RioCan":

        debt_inputs = [
            "bank_loans",
            "debentures_payable",
            "mortgages_payable"
        ]

        if "mortgages_held_for_sale" in values:
            debt_inputs.append("mortgages_held_for_sale")

        add_derived(
            "total_debt",
            sum(values[m] for m in debt_inputs),
            "CAD million",
            debt_inputs,
            " + ".join(debt_inputs)
        )

        cost_inputs = [
            "recoverable_operating_costs",
            "non_recoverable_operating_costs"
        ]

        add_derived(
            "property_operating_costs",
            sum(values[m] for m in cost_inputs),
            "CAD million",
            cost_inputs,
            " + ".join(cost_inputs)
        )

    rental_income = (
        values["rental_revenue_and_other"]
        + values["property_operating_costs"]
    )

    rental_inputs = [
        "rental_revenue_and_other",
        "recoverable_operating_costs",
        "non_recoverable_operating_costs"
    ] if company == "RioCan" else [
        "rental_revenue_and_other",
        "property_operating_costs"
    ]

    add_derived(
        "net_rental_income_and_other",
        rental_income,
        "CAD million",
        rental_inputs,
        " + ".join(rental_inputs)
    )

    add_derived(
        "book_debt_to_assets_ratio",
        values["total_debt"] / values["total_assets"] * 100,
        "Percent",
        debt_inputs + ["total_assets"]
        if company == "RioCan"
        else ["total_debt", "total_assets"],
        "total_debt / total_assets * 100"
    )

    add_derived(
        "net_rental_income_margin",
        rental_income / values["rental_revenue_and_other"] * 100,
        "Percent",
        rental_inputs,
        "net_rental_income_and_other / rental_revenue_and_other * 100"
    )

peer_derived_final = pd.DataFrame(derived_records)

# Correct NOI classification

peer_reit_corrected = peer_reit_annual_final.copy()

mask = (
    peer_reit_corrected["company"].eq("Choice Properties")
    & peer_reit_corrected["metric"].eq("noi_proportionate")
)

peer_reit_corrected.loc[
    mask, "metric"
] = "noi_accounting_basis"

# Final annual dataset

peer_annual_final = pd.concat(
    [
        peer_accounting,
        peer_derived_final,
        peer_reit_corrected
    ],
    ignore_index=True
)

peer_annual_final = peer_annual_final.sort_values(
    ["company", "period_end", "metric"]
).reset_index(drop=True)

print("Final Annual Shape:", peer_annual_final.shape)

print("\nRecords per Company and Year:")

print(
    peer_annual_final.groupby(
        ["company", "period_end"]
    ).size()
)

Final Annual Shape: (177, 14)

Records per Company and Year:
company            period_end
Choice Properties  2023-12-31    25
                   2024-12-31    29
                   2025-12-31    29
RioCan             2023-12-31    30
                   2024-12-31    32
                   2025-12-31    32
dtype: int64


/var/folders/s4/6pbd3b093t37sb7f1bds9vyw0000gn/T/ipykernel_83697/1604172349.py:189: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  peer_annual_final = pd.concat(


In [68]:
# Step 29 - Final annual validation and export

from pathlib import Path

required_columns = [
    "company",
    "ticker",
    "period_end",
    "period_type",
    "metric",
    "value",
    "unit",
    "source_file",
    "extraction_method"
]

duplicate_keys = peer_annual_final.duplicated(
    primary_keys
).sum()

missing_values = peer_annual_final[
    required_columns
].isna().sum().sum()

unit_conflicts = (
    peer_annual_final.groupby("metric")["unit"]
    .nunique() > 1
).sum()

print("Total Records:", len(peer_annual_final))
print("Total Columns:", len(peer_annual_final.columns))
print("Duplicate Primary Keys:", duplicate_keys)
print("Missing Required Values:", missing_values)
print("Unit Conflicts:", unit_conflicts)

# Accounting validation

accounting_checks = []

for (company, period), group in peer_annual_final.groupby(
    ["company", "period_end"]
):

    v = group.set_index("metric")["value"]

    balance_check = np.isclose(
        v["total_assets"],
        v["total_liabilities"] + v["total_equity"],
        atol=0.001,
        rtol=0
    )

    rental_check = np.isclose(
        v["rental_revenue_and_other"]
        + v["property_operating_costs"],
        v["net_rental_income_and_other"],
        atol=0.001,
        rtol=0
    )

    accounting_checks.append({
        "company": company,
        "period_end": period,
        "balance_sheet_check": balance_check,
        "net_rental_income_check": rental_check
    })

validation_report = pd.DataFrame(accounting_checks)

display(validation_report)

validation_passed = (
    duplicate_keys == 0
    and missing_values == 0
    and unit_conflicts == 0
    and validation_report[
        [
            "balance_sheet_check",
            "net_rental_income_check"
        ]
    ].all().all()
)

print("Structural Validation:", validation_passed)

assert validation_passed, "Review validation errors."

# Export cleaned dataset

processed_dir = base_dir / "processed"

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    processed_dir /
    "peer_annual_financials_cleaned.csv"
)

peer_annual_final.to_csv(
    output_file,
    index=False
)

saved_data = pd.read_csv(output_file)

print("\nExport Successful:", output_file.exists())
print("Saved Shape:", saved_data.shape)
print("Saved to:", output_file)

print(
    "Export Validation:",
    saved_data.shape == peer_annual_final.shape
)

Total Records: 177
Total Columns: 14
Duplicate Primary Keys: 0
Missing Required Values: 0
Unit Conflicts: 0


,company,period_end,balance_sheet_check,net_rental_income_check
0,Choice Properties,2023-12-31,True,True
1,Choice Properties,2024-12-31,True,True
2,Choice Properties,2025-12-31,True,True
3,RioCan,2023-12-31,True,True
4,RioCan,2024-12-31,True,True
5,RioCan,2025-12-31,True,True


Structural Validation: True

Export Successful: True
Saved Shape: (177, 14)
Saved to: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed/peer_annual_financials_cleaned.csv
Export Validation: True


In [70]:
# Step 30 - Locate quarterly financial statements

import pdfplumber

peer_quarterly_files = [
    file for file in peer_files
    if "2026" in file.name
]

print("Quarterly Reports:", len(peer_quarterly_files))

quarterly_page_texts = {}

for file in peer_quarterly_files:

    company = (
        "RioCan" if "RIO Report" in str(file)
        else "Choice Properties"
    )

    print("\n" + "=" * 70)
    print(company)
    print(file.name)
    print("=" * 70)

    with pdfplumber.open(file) as pdf:

        pages = {
            i: page.extract_text(layout=True) or ""
            for i, page in enumerate(pdf.pages, start=1)
        }

    quarterly_page_texts[(company, file.name)] = pages

    keywords = [
        "consolidated balance sheets",
        "consolidated statements of financial position",
        "consolidated statements of income",
        "consolidated statements of comprehensive income",
        "total assets",
        "total liabilities",
        "rental revenue"
    ]

    for keyword in keywords:

        matched = [
            p for p, text in pages.items()
            if keyword.lower() in
            " ".join(text.lower().split())
        ]

        print(f"{keyword}: {matched}")

Quarterly Reports: 6

RioCan
REI-Q1-2026-Report-to-Unitholders.pdf
consolidated balance sheets: [57, 79, 80, 85, 100, 101, 103, 104]
consolidated statements of financial position: []
consolidated statements of income: [58, 79, 81, 103]
consolidated statements of comprehensive income: [79, 82]
total assets: [20, 29, 46, 53, 57, 80, 88, 100]
total liabilities: [57, 80, 100]
rental revenue: [6, 7, 16, 21, 23, 48, 50, 52, 53, 58, 65, 81, 86, 98]

RioCan
REI-Q2-2026-Report-to-Unitholders.pdf
consolidated balance sheets: [61, 86, 87, 92, 109, 110, 111, 112]
consolidated statements of financial position: []
consolidated statements of income: [62, 86, 88, 111]
consolidated statements of comprehensive income: [86, 89]
total assets: [19, 32, 49, 56, 61, 87, 95, 109]
total liabilities: [61, 87, 109]
rental revenue: [5, 15, 20, 21, 23, 27, 51, 53, 55, 56, 62, 63, 71, 88, 93, 106]

RioCan
REI-Q3-2026-Report-to-Unitholders.pdf
consolidated balance sheets: [61, 86, 87, 92, 109, 110, 112]
consolidated

In [72]:
# Step 31 - Extract quarterly financial statement pages

quarterly_statement_mapping = {

    ("RioCan", "REI-Q1-2026-Report-to-Unitholders.pdf"):
        {"quarter": "Q1", "balance_sheet": 80, "income_statement": 81},

    ("RioCan", "REI-Q2-2026-Report-to-Unitholders.pdf"):
        {"quarter": "Q2", "balance_sheet": 87, "income_statement": 88},

    ("RioCan", "REI-Q3-2026-Report-to-Unitholders.pdf"):
        {"quarter": "Q3", "balance_sheet": 87, "income_statement": 88},

    ("Choice Properties", "Choice Properties Q1 2026 Report.pdf"):
        {"quarter": "Q1", "balance_sheet": 65, "income_statement": 66},

    ("Choice Properties", "Choice Properties Q2 2026 Report.pdf"):
        {"quarter": "Q2", "balance_sheet": 70, "income_statement": 71},

    ("Choice Properties", "Choice Properties Q3 2026 Report.pdf"):
        {"quarter": "Q3", "balance_sheet": 71, "income_statement": 72}
}

for key, mapping in quarterly_statement_mapping.items():

    company, file_name = key

    for statement in ["balance_sheet", "income_statement"]:

        page = mapping[statement]

        print("\n" + "=" * 80)
        print(company, "|", mapping["quarter"])
        print(statement, "| PDF Page:", page)
        print("=" * 80)

        print(quarterly_page_texts[key][page])


RioCan | Q1
balance_sheet | PDF Page: 80
                                                                                    
                                                                                    
       RIOCAN REAL ESTATE INVESTMENT TRUST                                          
       UNAUDITED INTERIM CONDENSED CONSOLIDATED BALANCE SHEETS                      
       (In thousands of Canadian dollars)                                           
                                                                                    
        As at                                  Note   March 31, 2026 December 31, 2025
        Assets                                                                      
        Investment properties                   3 $     13,598,006 $  13,628,959    
        Equity-accounted investments            4         161,403      159,596      
        Residential inventory                   5         235,633      236,745      
        Mortgages and

In [74]:
# Step 32.1 - Quarterly extraction function

import re
import pandas as pd
import numpy as np

number_pattern = r"\(?\s*-?\d[\d,]*(?:\.\d+)?\s*\)?|[—–]"

def parse_quarterly_number(token):

    token = token.strip().replace(",", "").replace(" ", "")

    if token in ["—", "–"]:
        return 0.0

    if token.startswith("(") and token.endswith(")"):
        return -float(token[1:-1])

    return float(token)


def extract_quarterly_value(text, label, n_columns):

    matches = []

    for line in text.splitlines():

        line = " ".join(line.strip().split())

        if not line.lower().startswith(label.lower()):
            continue

        remaining = line[len(label):].strip()

        # Prevent matching longer metric labels
        if remaining and remaining[0].isalpha():
            continue

        tokens = re.findall(number_pattern, remaining)

        if len(tokens) < n_columns:
            continue

        values = [
            parse_quarterly_number(token)
            for token in tokens[-n_columns:]
        ]

        matches.append({
            "values": values,
            "original_line": line
        })

    if not matches:
        raise ValueError(f"{label}: No matching row")

    if any(
        m["values"] != matches[0]["values"]
        for m in matches
    ):
        raise ValueError(f"{label}: Conflicting rows")

    return {
        "value": matches[0]["values"][0],
        "original_line": matches[0]["original_line"]
    }


print("Quarterly Extraction Function: Ready")

Quarterly Extraction Function: Ready


In [76]:
# Step 32.2 - Quarterly metric mapping

quarterly_metric_mapping = {

    "RioCan": {

        "balance_sheet": {
            "total_assets": "Total assets",
            "total_liabilities": "Total liabilities",
            "total_equity": "Unitholders' equity",
            "investment_properties": "Investment properties",
            "cash_and_cash_equivalents": "Cash and cash equivalents",
            "debentures_payable": "Debentures payable",
            "mortgages_payable": "Mortgages payable",
            "mortgages_held_for_sale":
                "Mortgages payable associated with assets held for sale",
            "bank_loans": "Lines of credit and other bank loans"
        },

        "income_statement": {
            "rental_revenue_and_other": "Rental revenue",
            "recoverable_operating_costs":
                "Recoverable under tenant leases",
            "non_recoverable_operating_costs":
                "Non-recoverable costs",
            "interest_costs_net": "Interest costs, net",
            "net_income": "Net income"
        }
    },

    "Choice Properties": {

        "balance_sheet": {
            "total_assets": "Total Assets",
            "total_liabilities": "Total Liabilities",
            "total_equity": "Total Equity",
            "investment_properties": "Investment properties",
            "cash_and_cash_equivalents": "Cash and cash equivalents",
            "long_term_debt": "Long term debt",
            "credit_facility": "Credit facility",
            "exchangeable_units_liability": "Exchangeable Units"
        },

        "income_statement": {
            "rental_revenue_and_other": "Rental revenue",
            "property_operating_costs": "Property operating costs",
            "net_financing_charges":
                "Net interest expense and other financing charges",
            "net_income": "Net Income"
        }
    }
}

print("Quarterly Metric Mapping: Ready")

Quarterly Metric Mapping: Ready


In [78]:
# Step 32.3 - Batch quarterly financial extraction

quarter_end_mapping = {
    "Q1": "2026-03-31",
    "Q2": "2026-06-30",
    "Q3": "2026-09-30"
}

ticker_mapping = {
    "RioCan": "REI-UN.TO",
    "Choice Properties": "CHP-UN.TO"
}

negative_expense_metrics = {
    "recoverable_operating_costs",
    "non_recoverable_operating_costs",
    "property_operating_costs",
    "interest_costs_net",
    "net_financing_charges"
}

quarterly_records = []
quarterly_errors = []

for key, mapping in quarterly_statement_mapping.items():

    company, file_name = key
    quarter = mapping["quarter"]

    for statement, metrics in (
        quarterly_metric_mapping[company].items()
    ):

        page = mapping[statement]
        text = quarterly_page_texts[key][page]

        # Balance sheets have two date columns.
        # Q2 and Q3 income statements have four columns.

        n_columns = (
            2 if statement == "balance_sheet"
            or quarter == "Q1"
            else 4
        )

        for metric, label in metrics.items():

            # Handle different net income labels

            if metric == "net_income":

                if company == "RioCan":
                    label = (
                        "Net income"
                        if quarter == "Q2"
                        else "Net income (loss)"
                    )

                elif company == "Choice Properties":
                    label = (
                        "Net Loss"
                        if quarter in ["Q1", "Q2"]
                        else "Net Income (Loss)"
                    )

            try:

                result = extract_quarterly_value(
                    text,
                    label,
                    n_columns
                )

                original_value = result["value"]
                value = original_value / 1000

                if metric in negative_expense_metrics:
                    value = -abs(value)

                quarterly_records.append({

                    "company": company,
                    "ticker": ticker_mapping[company],
                    "period_end": quarter_end_mapping[quarter],
                    "period_type": "Quarterly",
                    "metric": metric,
                    "value": value,
                    "unit": "CAD million",
                    "source_file": file_name,
                    "source_page": page,
                    "source_label": label,
                    "original_value": original_value,
                    "original_unit": "CAD thousand",
                    "extraction_method": "Text-based extraction",
                    "original_line": result["original_line"]
                })

            except ValueError as e:

                quarterly_errors.append({

                    "company": company,
                    "quarter": quarter,
                    "metric": metric,
                    "source_page": page,
                    "error": str(e)
                })

peer_quarterly_raw = pd.DataFrame(quarterly_records)

print("Extracted Shape:", peer_quarterly_raw.shape)
print("Extraction Errors:", len(quarterly_errors))

if quarterly_errors:
    display(pd.DataFrame(quarterly_errors))

display(
    peer_quarterly_raw.pivot(
        index=["company", "metric"],
        columns="period_end",
        values="value"
    ).round(3)
)

Extracted Shape: (78, 14)
Extraction Errors: 0


period_end                                         2026-03-31  2026-06-30  \
company           metric                                                    
Choice Properties cash_and_cash_equivalents            34.759      45.742   
                  credit_facility                      28.562      67.930   
                  exchangeable_units_liability      6,079.281   6,467.151   
                  investment_properties            15,850.000  16,011.000   
                  long_term_debt                    6,788.389   6,719.101   
                  net_financing_charges              -150.063    -151.012   
                  net_income                          -87.194    -176.381   
                  property_operating_costs           -105.095    -105.300   
                  rental_revenue_and_other            361.192     361.331   
                  total_assets                     17,724.833  17,960.092   
                  total_equity                      4,434.568   4,191.890   
                  total_liabilities                13,290.265  13,768.202   
RioCan            bank_loans                          622.409   1,106.879   
                  cash_and_cash_equivalents            70.215      66.630   
                  debentures_payable                4,438.732   3,939.514   
                  interest_costs_net                  -71.909     -71.143   
                  investment_properties            13,598.006  13,605.045   
                  mortgages_held_for_sale             178.824      75.594   
                  mortgages_payable                 2,047.117   1,869.708   
                  net_income                           93.162     151.237   
                  non_recoverable_operating_costs      -9.476      -8.537   
                  recoverable_operating_costs        -118.489    -102.256   
                  rental_revenue_and_other            308.261     296.802   
                  total_assets                     14,934.480  14,673.472   
                  total_equity                      7,108.728   7,177.174   
                  total_liabilities                 7,825.752   7,496.298   

period_end                                         2026-09-30  
company           metric                                       
Choice Properties cash_and_cash_equivalents           356.543  
                  credit_facility                      27.555  
                  exchangeable_units_liability      5,869.514  
                  investment_properties            16,150.000  
                  long_term_debt                    7,012.858  
                  net_financing_charges              -151.986  
                  net_income                          888.506  
                  property_operating_costs            -98.336  
                  rental_revenue_and_other            366.327  
                  total_assets                     18,519.389  
                  total_equity                      5,018.967  
                  total_liabilities                13,500.422  
RioCan            bank_loans                        1,100.958  
                  cash_and_cash_equivalents            49.478  
                  debentures_payable                3,940.283  
                  interest_costs_net                  -69.570  
                  investment_properties            13,797.083  
                  mortgages_held_for_sale               0.000  
                  mortgages_payable                 1,859.935  
                  net_income                          122.603  
                  non_recoverable_operating_costs      -8.173  
                  recoverable_operating_costs        -104.657  
                  rental_revenue_and_other            300.242  
                  total_assets                     14,611.872  
                  total_equity                      7,223.235  
                  total_liabilities                 7,388.637

In [80]:
# Step 33.1 - Calculate quarterly derived metrics

peer_quarterly_base = peer_quarterly_raw.copy()

peer_quarterly_base["metric"] = (
    peer_quarterly_base["metric"].replace({
        "rental_revenue_and_other": "rental_revenue"
    })
)

derived_records = []

for (company, period), group in peer_quarterly_base.groupby(
    ["company", "period_end"]
):

    values = group.set_index("metric")["value"].to_dict()

    if company == "RioCan":

        debt_metrics = [
            "debentures_payable",
            "mortgages_payable",
            "mortgages_held_for_sale",
            "bank_loans"
        ]

        total_debt = sum(values[m] for m in debt_metrics)

        operating_costs = (
            values["recoverable_operating_costs"]
            + values["non_recoverable_operating_costs"]
        )

    else:

        debt_metrics = [
            "long_term_debt",
            "credit_facility"
        ]

        total_debt = sum(values[m] for m in debt_metrics)

        operating_costs = values["property_operating_costs"]

    net_rental_income = (
        values["rental_revenue"] + operating_costs
    )

    debt_ratio = (
        total_debt / values["total_assets"] * 100
    )

    rental_margin = (
        net_rental_income / values["rental_revenue"] * 100
    )

    calculated_metrics = [
        (
            "total_debt",
            total_debt,
            "CAD million",
            " + ".join(debt_metrics),
            "total_assets"
        ),
        (
            "net_rental_income",
            net_rental_income,
            "CAD million",
            "rental_revenue + property_operating_costs",
            "rental_revenue"
        ),
        (
            "book_debt_to_assets_ratio",
            debt_ratio,
            "Percent",
            "total_debt / total_assets * 100",
            "total_assets"
        ),
        (
            "net_rental_income_margin",
            rental_margin,
            "Percent",
            "net_rental_income / rental_revenue * 100",
            "rental_revenue"
        )
    ]

    if company == "RioCan":

        calculated_metrics.append((
            "property_operating_costs",
            operating_costs,
            "CAD million",
            "recoverable_operating_costs + non_recoverable_operating_costs",
            "rental_revenue"
        ))

    for metric, value, unit, formula, source_metric in calculated_metrics:

        source = group.loc[
            group["metric"] == source_metric
        ].iloc[0]

        record = source.to_dict()

        record.update({
            "metric": metric,
            "value": round(value, 6),
            "unit": unit,
            "source_label": formula,
            "original_value": np.nan,
            "original_unit": None,
            "extraction_method": "Calculated",
            "original_line": formula
        })

        derived_records.append(record)

peer_quarterly_derived = pd.DataFrame(
    derived_records
)

print("Derived Shape:", peer_quarterly_derived.shape)

display(
    peer_quarterly_derived.pivot(
        index=["company", "metric"],
        columns="period_end",
        values="value"
    ).round(3)
)

Derived Shape: (27, 14)


period_end                                   2026-03-31  2026-06-30  \
company           metric                                              
Choice Properties book_debt_to_assets_ratio      38.460      37.790   
                  net_rental_income             256.097     256.031   
                  net_rental_income_margin       70.903      70.858   
                  total_debt                  6,816.951   6,787.031   
RioCan            book_debt_to_assets_ratio      48.794      47.649   
                  net_rental_income             180.296     186.009   
                  net_rental_income_margin       58.488      62.671   
                  property_operating_costs     -127.965    -110.793   
                  total_debt                  7,287.082   6,991.695   

period_end                                   2026-09-30  
company           metric                                 
Choice Properties book_debt_to_assets_ratio      38.016  
                  net_rental_income             267.991  
                  net_rental_income_margin       73.156  
                  total_debt                  7,040.413  
RioCan            book_debt_to_assets_ratio      47.230  
                  net_rental_income             187.412  
                  net_rental_income_margin       62.420  
                  property_operating_costs     -112.830  
                  total_debt                  6,901.176

In [82]:
# Step 33.2 - Final quarterly data validation

peer_quarterly_final = pd.concat(
    [
        peer_quarterly_base,
        peer_quarterly_derived
    ],
    ignore_index=True
)

# Match period type with SmartCentres quarterly dataset

smart_quarterly_reference = pd.read_csv(
    base_dir / "processed" /
    "smartcentres_quarterly_financials_cleaned.csv"
)

period_types = smart_quarterly_reference[
    "period_type"
].dropna().unique()

assert len(period_types) == 1

peer_quarterly_final["period_type"] = period_types[0]

primary_key = [
    "ticker",
    "period_end",
    "period_type",
    "metric"
]

peer_quarterly_final = (
    peer_quarterly_final
    .sort_values(primary_key)
    .reset_index(drop=True)
)

duplicates = peer_quarterly_final.duplicated(
    primary_key
).sum()

required_columns = [
    "company", "ticker", "period_end",
    "period_type", "metric", "value",
    "unit", "source_file", "source_page",
    "extraction_method"
]

missing_required = peer_quarterly_final[
    required_columns
].isna().sum().sum()

unit_conflicts = (
    peer_quarterly_final
    .groupby("metric")["unit"]
    .nunique()
    .gt(1)
    .sum()
)

validation_records = []

for (company, period), group in peer_quarterly_final.groupby(
    ["company", "period_end"]
):

    v = group.set_index("metric")["value"].to_dict()

    balance_check = np.isclose(
        v["total_assets"],
        v["total_liabilities"] + v["total_equity"],
        atol=0.001,
        rtol=0
    )

    rental_check = np.isclose(
        v["net_rental_income"],
        v["rental_revenue"] + v["property_operating_costs"],
        atol=0.001,
        rtol=0
    )

    validation_records.append({
        "company": company,
        "period_end": period,
        "balance_sheet_check": balance_check,
        "net_rental_income_check": rental_check
    })

quarterly_validation = pd.DataFrame(
    validation_records
)

structural_validation = bool(
    peer_quarterly_final.shape == (105, 14)
    and duplicates == 0
    and missing_required == 0
    and unit_conflicts == 0
    and len(quarterly_validation) == 6
    and quarterly_validation[
        [
            "balance_sheet_check",
            "net_rental_income_check"
        ]
    ].all().all()
)

print("Final Shape:", peer_quarterly_final.shape)
print("Duplicate Primary Keys:", duplicates)
print("Missing Required Values:", missing_required)
print("Unit Conflicts:", unit_conflicts)

display(quarterly_validation)

print("\nRecords per Company and Quarter:")

print(
    peer_quarterly_final.groupby(
        ["company", "period_end"]
    ).size()
)

print("\nStructural Validation:", structural_validation)

Final Shape: (105, 14)
Duplicate Primary Keys: 0
Missing Required Values: 0
Unit Conflicts: 0


,company,period_end,balance_sheet_check,net_rental_income_check
0,Choice Properties,2026-03-31,True,True
1,Choice Properties,2026-06-30,True,True
2,Choice Properties,2026-09-30,True,True
3,RioCan,2026-03-31,True,True
4,RioCan,2026-06-30,True,True
5,RioCan,2026-09-30,True,True



Records per Company and Quarter:
company            period_end
Choice Properties  2026-03-31    16
                   2026-06-30    16
                   2026-09-30    16
RioCan             2026-03-31    19
                   2026-06-30    19
                   2026-09-30    19
dtype: int64

Structural Validation: True


In [84]:
# Step 34 - Export cleaned peer quarterly financials

assert structural_validation, "Validation failed."

output_dir = base_dir / "processed"

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

output_file = (
    output_dir /
    "peer_quarterly_financials_cleaned.csv"
)

peer_quarterly_final.to_csv(
    output_file,
    index=False
)

saved_data = pd.read_csv(output_file)

print("Export Successful:", output_file.exists())
print("Saved Shape:", saved_data.shape)
print("Saved to:", output_file)

print(
    "Export Validation:",
    saved_data.shape == peer_quarterly_final.shape
    and saved_data.columns.tolist()
    == peer_quarterly_final.columns.tolist()
    and np.allclose(
        saved_data["value"],
        peer_quarterly_final["value"],
        atol=0.000001,
        rtol=0
    )
)

Export Successful: True
Saved Shape: (105, 14)
Saved to: /Users/yuanquan0901/Desktop/8730 _ Group Assignment/processed/peer_quarterly_financials_cleaned.csv
Export Validation: True
